# History Matching for Joint Calibration

Wave-by-wave history matching with Gaussian-process emulators (Craig, Goldstein,
Smith; adapted for TB by Andrews 2014, McCreesh & White 2018). The procedure:

1. Fit a GP emulator per output target on the current calibration DB.
2. Compute **implausibility** at a dense candidate grid in parameter space.
3. Retain only the NROY (Not Ruled Out Yet) region: points where the max
   implausibility across all targets is below a threshold (typically 3).
4. Design the next wave inside NROY (max-min LHS).
5. Run those points on COMPS, parse outputs, append to DB.
6. Repeat until NROY is small and stable, OR until you've narrowed it to a
   plausible region for the paper.

Each wave should shrink NROY substantially. Typical: 3-4 waves to convergence,
with wave sizes of 15-30 points.

**Dependencies:** `mogp_emulator` (Exeter), `scipy`, `scikit-learn`. Install via:

```
pip install mogp-emulator scikit-learn scipy
```


## 1. Imports + framework


In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import everything we need from the framework notebook by re-executing its
# definitions inline. (Notebooks don't import each other natively; we mirror
# the key pieces here for reproducibility.)
%run calibration_framework.ipynb

# Try emulator package; fall back to sklearn GPs if mogp isn't installed.
try:
    import mogp_emulator as mogp
    EMULATOR_BACKEND = 'mogp'
except ImportError:
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
    EMULATOR_BACKEND = 'sklearn'
    print('mogp_emulator not installed; using sklearn GP fallback.')

print(f'Emulator backend: {EMULATOR_BACKEND}')
print(f'DB rows: {len(load_db())}')


Project root: C:\Users\young\OneDrive\Documents\GitHub\TB_India_DTK
Calibration DB: C:\Users\young\OneDrive\Documents\GitHub\TB_India_DTK\calibration_db.parquet (exists=False)
Parameter space: 5 dims
  beta                range=[0.01, 0.06]  (log)
  tb_duration_yr      range=[1.5, 3.0]  (linear)
  presymp_yr          range=[0.15, 0.5]  (linear)
  cs_fast_d           range=[30, 180]  (log)
  cs_slow_d           range=[90, 500]  (log)
10 calibration targets:
  burden   : 3
  fractions: 3
  tests    : 4

Diagnostic outputs (saved but not calibrated): 4
Example design (no NROY filter):


,beta,tb_duration_yr,presymp_yr,cs_fast_d,cs_slow_d
0,0.010,2.727,0.439,38.425,181.504
1,0.039,1.501,0.167,119.190,311.120
2,0.017,2.347,0.343,129.081,108.667
3,0.058,1.821,0.289,47.036,420.029
4,0.028,2.572,0.373,64.792,174.497


Parser ready.
mogp_emulator not installed; using sklearn GP fallback.
Emulator backend: sklearn
DB rows: 0


## 2. Build emulator inputs

Map DB rows to (X, Y) per output target. X is the normalized parameter matrix
(each row in [0,1]^d); Y is a vector of one output across all design points.


In [2]:
def db_to_emulator_inputs(db: pd.DataFrame) -> tuple[np.ndarray, dict]:
    """
    Returns X (n_points, d) of normalized params, and Y_dict {target: y_vec}.
    Drops rows with NaN outputs.
    """
    valid = db.dropna(subset=TARGET_NAMES, how='any').copy()
    if len(valid) == 0:
        raise ValueError('No valid rows in DB; run seed_db_from_manual_runs() first.')
    X_rows = []
    for _, row in valid.iterrows():
        X_rows.append([PARAM_SPACE[i].normalize(row[p.name])
                       for i, p in enumerate(PARAM_SPACE)])
    X = np.array(X_rows)
    Y = {t.name: valid[t.name].to_numpy() for t in TARGETS}
    return X, Y


## 3. Fit emulators

One GP per target. We use `mogp_emulator.GaussianProcess` with Matern 5/2
kernel + heteroscedastic noise. sklearn fallback uses the same kernel
structure.


In [3]:
def fit_emulators(X: np.ndarray, Y: dict) -> dict:
    """
    Returns dict of {target: fitted_emulator}.
    Each emulator exposes .predict(X_test) -> (mean, var).
    """
    emulators = {}
    for tname, y in Y.items():
        if EMULATOR_BACKEND == 'mogp':
            gp = mogp.GaussianProcess(X, y, mean='constant', kernel='Matern52',
                                       nugget='adaptive')
            gp = mogp.fit_GP_MAP(gp)
            emulators[tname] = gp
        else:
            kernel = (ConstantKernel(1.0, (1e-3, 1e3))
                      * Matern(length_scale=[0.3]*X.shape[1], nu=2.5)
                      + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e1)))
            gp = GaussianProcessRegressor(kernel=kernel,
                                          normalize_y=True,
                                          n_restarts_optimizer=5)
            gp.fit(X, y)
            emulators[tname] = gp
    return emulators


def emulator_predict(emulator, X_test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Returns (mean, variance) at X_test."""
    if EMULATOR_BACKEND == 'mogp':
        result = emulator.predict(X_test)
        return result.mean, result.unc**2
    else:
        mean, std = emulator.predict(X_test, return_std=True)
        return mean, std**2


## 4. Implausibility + NROY

For each candidate point x, implausibility w.r.t. target i is:
$$ I_i(x) = \frac{|E[f_i(x)] - z_i|}{\sqrt{\text{Var}_{\text{emul}}[f_i(x)] + \sigma_{\text{obs},i}^2 + \sigma_{\text{disc},i}^2}} $$

Maximum implausibility across i is the screening criterion. We retain x in
NROY iff $\max_i I_i(x) < 3$ (standard cutoff; 3-sigma).


In [4]:
IMPLAUSIBILITY_CUTOFF = 3.0


def implausibility_grid(emulators: dict, X_grid: np.ndarray) -> np.ndarray:
    """
    Compute max implausibility across all targets for each row in X_grid.
    Returns (n_points,) array of max implausibility.
    """
    target_by_name = {t.name: t for t in TARGETS}
    Imax = np.zeros(X_grid.shape[0])
    for tname, emu in emulators.items():
        mu, var = emulator_predict(emu, X_grid)
        tgt = target_by_name[tname]
        total_var = var + tgt.sigma_total**2
        I_i = np.abs(mu - tgt.z) / np.sqrt(total_var)
        Imax = np.maximum(Imax, I_i)
    return Imax


def nroy_mask_factory(emulators: dict,
                     cutoff: float = IMPLAUSIBILITY_CUTOFF):
    """
    Returns a callable: nroy_mask(X) -> boolean array of NROY membership.
    """
    def mask(X: np.ndarray) -> np.ndarray:
        return implausibility_grid(emulators, X) < cutoff
    return mask


## 5. Run wave 1: initial Latin-hypercube design

If the DB is empty (or only has v0 manual run), wave 1 is an unconstrained LHS
over the full parameter ranges. 15-20 points recommended.

After running, you save the design, submit on COMPS, parse results, and append
to the DB. This cell only generates the design and writes the JSON for the
sweep -- COMPS execution is manual.


In [5]:
WAVE = 1
N_POINTS = 15

db = load_db()
print(f'DB rows before wave {WAVE}: {len(db)}')

if WAVE == 1 or len(db.dropna(subset=TARGET_NAMES, how='any')) < 5:
    # Unconstrained LHS
    design = generate_design(n_points=N_POINTS, seed=WAVE)
    print(f'Wave {WAVE}: unconstrained LHS, {N_POINTS} points.')
else:
    # Fit emulators on current DB, generate NROY-restricted design
    X, Y = db_to_emulator_inputs(db)
    print(f'Fitting emulators on {len(X)} DB rows...')
    emulators = fit_emulators(X, Y)
    nroy_mask = nroy_mask_factory(emulators)
    design = generate_design(n_points=N_POINTS, nroy_mask=nroy_mask, seed=WAVE)
    print(f'Wave {WAVE}: NROY-restricted LHS, {len(design)} points.')

save_design_for_sweep(design, wave=WAVE)
display(design.round(3))


DB rows before wave 1: 0
Wave 1: unconstrained LHS, 15 points.
Wrote C:\Users\young\OneDrive\Documents\GitHub\TB_India_DTK\design_wave1.json  (15 design points)


,beta,tb_duration_yr,presymp_yr,cs_fast_d,cs_slow_d
0,0.034,1.983,0.252,36.666,157.643
1,0.021,2.685,0.200,105.506,216.015
2,0.016,2.890,0.459,125.332,171.990
3,0.020,2.420,0.488,147.941,484.890
4,0.038,2.396,0.267,96.189,232.559
5,0.013,2.259,0.228,162.930,426.024
6,0.048,2.988,0.320,127.675,93.073
7,0.012,2.529,0.366,54.517,301.264
8,0.044,1.728,0.167,54.078,358.021
9,0.030,2.107,0.388,69.918,101.778


## 6. After COMPS runs complete: ingest results

Manually point at the new experiment folder(s). Each wave can be one COMPS
experiment with a sweep over `subsample`, or multiple experiments if you
submitted them piecemeal.


In [6]:
def ingest_wave(wave: int, experiment_uids: list[str], design: pd.DataFrame):
    """
    Parse each completed experiment in this wave and append to DB.
    `experiment_uids[i]` corresponds to design point i.
    """
    rows = []
    for i, uid in enumerate(experiment_uids):
        exp_dir = PROJECT_ROOT / uid
        if not exp_dir.exists():
            print(f'  SKIP point {i} ({uid}): not found on disk')
            continue
        try:
            outputs = parse_experiment_dir(exp_dir)
        except Exception as e:
            print(f'  PARSE FAIL point {i}: {e}')
            continue
        params = {p.name: float(design.iloc[i][p.name]) for p in PARAM_SPACE}
        row = make_row(wave=wave, point_id=f'wave{wave}_pt{i}',
                       method='history_matching', experiment_uid=uid,
                       params=params, outputs=outputs)
        rows.append(row)
    return append_to_db(rows)


# Example call (uncomment + fill in once your COMPS sweep is done):
# ingested_db = ingest_wave(
#     wave=1,
#     experiment_uids=['uid_for_point_0', 'uid_for_point_1', ...],
#     design=design,
# )


## 7. Wave-by-wave NROY contraction plot

Visualize how each wave shrinks NROY in 2D projections. Pairs plot of the
parameter space, color-coded by max-implausibility, with wave-N design points
overlaid.


In [7]:
def plot_nroy_pairs(emulators: dict, db: pd.DataFrame,
                    n_grid_per_dim: int = 30,
                    save: str | None = None):
    """
    2D projections (X_i, X_j) showing min-over-other-dims implausibility.
    Overlays DB points colored by wave.
    """
    d = len(PARAM_SPACE)
    # Build dense candidate grid in [0,1]^d
    n_total = 5000
    sampler = qmc.LatinHypercube(d=d, seed=999)
    X_grid = sampler.random(n=n_total)
    Imax = implausibility_grid(emulators, X_grid)

    fig, axes = plt.subplots(d, d, figsize=(2.6 * d, 2.6 * d))
    waves = sorted(db['wave'].dropna().unique())
    cmap = plt.colormaps.get_cmap('tab10')
    wave_colors = {w: cmap(i) for i, w in enumerate(waves)}

    for i in range(d):
        for j in range(d):
            ax = axes[i, j]
            if i == j:
                ax.hist(X_grid[Imax < IMPLAUSIBILITY_CUTOFF, i], bins=20,
                        color='steelblue', alpha=0.6)
                ax.set_title(PARAM_SPACE[i].name, fontsize=8)
                continue
            keep = Imax < IMPLAUSIBILITY_CUTOFF
            ax.scatter(X_grid[keep, j], X_grid[keep, i], s=2,
                       c=Imax[keep], cmap='viridis_r', alpha=0.4)
            for w in waves:
                rows = db[db['wave'] == w].dropna(subset=TARGET_NAMES, how='any')
                if len(rows) == 0:
                    continue
                xn = [PARAM_SPACE[j].normalize(v) for v in rows[PARAM_SPACE[j].name]]
                yn = [PARAM_SPACE[i].normalize(v) for v in rows[PARAM_SPACE[i].name]]
                ax.scatter(xn, yn, s=40, c=[wave_colors[w]], edgecolor='black',
                           label=f'wave {int(w)}', zorder=3)
            if i == d - 1: ax.set_xlabel(PARAM_SPACE[j].name, fontsize=8)
            if j == 0:     ax.set_ylabel(PARAM_SPACE[i].name, fontsize=8)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    handles, labels = axes[0, 1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='upper right', fontsize=9)
    fig.suptitle(f'NROY (implausibility < {IMPLAUSIBILITY_CUTOFF}) -- '
                 f'normalized parameter space pairs',
                 fontsize=11, fontweight='bold', y=1.005)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches='tight')
        print(f'Saved: {save}')
    plt.show()


# Run when you have >= 5 valid DB rows:
# X, Y = db_to_emulator_inputs(load_db())
# emulators = fit_emulators(X, Y)
# plot_nroy_pairs(emulators, load_db(), save='Fig_nroy_wave1.png')


## 8. Convergence check

After each wave, evaluate:
- **NROY volume** (fraction of LHS candidates passing the implausibility cutoff)
- **Best in-DB implausibility** (`min(implausibility_max)` over all rows)
- **Emulator cross-validation** (leave-one-out RMSE per target)

Stop when NROY volume is small (~<5%), best implausibility is < 2-3, and
the emulator LOO-RMSE is below the target sigma_total. That's a 'plausibly
calibrated' state and the basis for the manuscript's claim.


In [8]:
def wave_summary(db: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for w, group in db.groupby('wave'):
        valid = group.dropna(subset=TARGET_NAMES, how='any')
        rows.append({
            'wave': int(w),
            'n_total':    len(group),
            'n_valid':    len(valid),
            'min_implausibility_max': valid['implausibility_max'].min() if len(valid) else np.nan,
            'median_implausibility_max': valid['implausibility_max'].median() if len(valid) else np.nan,
            'min_loss_burden': valid['loss_burden'].min() if len(valid) else np.nan,
            'min_loss_timing': valid['loss_timing'].min() if len(valid) else np.nan,
        })
    return pd.DataFrame(rows)


def nroy_volume(emulators: dict, n_samples: int = 20000, seed: int = 7) -> float:
    sampler = qmc.LatinHypercube(d=len(PARAM_SPACE), seed=seed)
    X = sampler.random(n=n_samples)
    Imax = implausibility_grid(emulators, X)
    return float(np.mean(Imax < IMPLAUSIBILITY_CUTOFF))


def loo_rmse(X: np.ndarray, Y: dict) -> dict:
    """Leave-one-out cross-validation RMSE per target."""
    out = {}
    for tname, y in Y.items():
        preds = np.zeros_like(y)
        for i in range(len(y)):
            mask = np.arange(len(y)) != i
            Xtr, ytr = X[mask], y[mask]
            emu = fit_emulators(Xtr, {tname: ytr})[tname]
            mu, _ = emulator_predict(emu, X[i:i+1])
            preds[i] = mu[0]
        out[tname] = float(np.sqrt(np.mean((preds - y)**2)))
    return out


# Run convergence summary after each wave:
# print('Wave summary:')
# display(wave_summary(load_db()))
# X, Y = db_to_emulator_inputs(load_db())
# emulators = fit_emulators(X, Y)
# print(f'NROY volume: {nroy_volume(emulators):.1%}')
# print('LOO-RMSE per target:'); display(pd.Series(loo_rmse(X, Y)))
